# Machine Learning Notes
## Day 28: Column Transformer — Answer Key

> **Watermark:** Amol Jagtap | amoljagtap3001@gmail.com  
> **Topic:** ColumnTransformer in Scikit-learn  
> **Difficulty:** Beginner to Intermediate  

---
### Note:
This notebook contains FULLY WORKED SOLUTIONS for every exercise in
**Day28_Column_Transformer_Practice_Questions.ipynb**. Use this to check your work
or to study the reference implementation.

### Topics Covered:
1. Identifying which transformation each column needs
2. Basic ColumnTransformer with multiple transformers
3. Using the remainder parameter
4. Nesting a Pipeline inside ColumnTransformer (impute + scale)
5. ColumnTransformer combined with a model in a full Pipeline
6. Selecting columns automatically with make_column_selector
7. Mini end-to-end project — mixed dataset preprocessing

---

In [ ]:
# ============================================================
# SETUP — Run this first!
# ============================================================
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import (StandardScaler, OneHotEncoder,
                                     OrdinalEncoder, LabelEncoder)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

print('All libraries imported successfully!')
print('Notebook by: Amol Jagtap | amoljagtap3001@gmail.com')

---
## Section 1: Identifying Which Transformation Each Column Needs

In [ ]:
# ============================================================
# ANSWER 1: Column Classification
# ============================================================

df = pd.DataFrame({
    'Age':          [25, 40, 33, 55, 29],
    'Salary':       [50000, 85000, np.nan, 120000, 60000],
    'City':         ['Mumbai', 'Delhi', 'Pune', 'Mumbai', 'Delhi'],
    'Education':    ['Graduate', 'School', 'Postgraduate', 'Graduate', 'School'],
    'Approved':     ['Yes', 'No', 'No', 'Yes', 'No']
})

transformation_plan = {
    'Age':        'StandardScaler (numerical, no missing values)',
    'Salary':     'SimpleImputer(mean) -> StandardScaler (numerical, has missing values)',
    'City':       'OneHotEncoder (nominal categorical — no natural order)',
    'Education':  'OrdinalEncoder (ordinal categorical — School < Graduate < Postgraduate)',
}

print('Transformation plan for each feature column:')
for col, plan in transformation_plan.items():
    print(f'  {col:12s} -> {plan}')

print('\nWhy \'Approved\' is excluded from ColumnTransformer:')
print('\'Approved\' is the TARGET variable (y), not an input feature.')
print('ColumnTransformer is designed to preprocess INPUT features (X) only.')
print('The target should be encoded separately, typically with LabelEncoder,')
print('and must never be included alongside X in the same ColumnTransformer.')

---
## Section 2: Basic ColumnTransformer with Multiple Transformers

In [ ]:
# ============================================================
# ANSWER 2: ColumnTransformer for Age, City, Education
# ============================================================

df2 = pd.DataFrame({
    'Age':        [25, 40, 33, 55, 29, 47],
    'City':       ['Mumbai', 'Delhi', 'Pune', 'Mumbai', 'Delhi', 'Pune'],
    'Education':  ['Graduate', 'School', 'Postgraduate',
                    'Graduate', 'School', 'Postgraduate']
})

edu_order = [['School', 'Graduate', 'Postgraduate']]

ct = ColumnTransformer(transformers=[
    ('age_scale', StandardScaler(), ['Age']),
    ('city_ohe',  OneHotEncoder(), ['City']),
    ('edu_ord',   OrdinalEncoder(categories=edu_order), ['Education'])
])

transformed = ct.fit_transform(df2)

print('Transformed array:')
print(transformed)

print('\nGenerated feature names:')
print(ct.get_feature_names_out())

result_df = pd.DataFrame(transformed, columns=ct.get_feature_names_out())
print('\nAs a readable DataFrame:')
print(result_df.round(3).to_string(index=False))

---
## Section 3: Using the remainder Parameter

In [ ]:
# ============================================================
# ANSWER 3: remainder='drop' vs remainder='passthrough'
# ============================================================

df3 = pd.DataFrame({
    'Age':     [25, 40, 33, 55],
    'City':    ['Mumbai', 'Delhi', 'Pune', 'Mumbai'],
    'Salary':  [50000, 85000, 60000, 120000],
    'Score':   [7.5, 8.2, 6.9, 9.1]
})

# 1 & 2: default remainder='drop'
ct_drop = ColumnTransformer(transformers=[
    ('city_ohe', OneHotEncoder(), ['City'])
])  # remainder defaults to 'drop'
result_drop = ct_drop.fit_transform(df3)
print('remainder="drop" (default):')
print(result_drop)
print('Shape:', result_drop.shape, '-> Age, Salary, Score were DROPPED')

# 3: remainder='passthrough'
ct_pass = ColumnTransformer(transformers=[
    ('city_ohe', OneHotEncoder(), ['City'])
], remainder='passthrough')
result_pass = ct_pass.fit_transform(df3)
print('\nremainder="passthrough":')
print(result_pass)
print('Shape:', result_pass.shape, '-> Age, Salary, Score were KEPT, appended at the end')

# 4: comparison
print(f'\nColumn count comparison: drop={result_drop.shape[1]} columns, '
      f'passthrough={result_pass.shape[1]} columns')
print('The 3 extra columns in "passthrough" are Age, Salary, Score,')
print('appended unchanged after the one-hot encoded City columns.')

---
## Section 4: Nesting a Pipeline Inside ColumnTransformer

In [ ]:
# ============================================================
# ANSWER 4: Pipeline (Impute + Scale) Nested in ColumnTransformer
# ============================================================

df4 = pd.DataFrame({
    'Age':     [25, 40, np.nan, 55, 29],
    'Fever':   [98.6, np.nan, 101.2, 99.5, np.nan],
    'Gender':  ['Male', 'Female', 'Female', 'Male', 'Female']
})

# 1. Pipeline for Age: impute (mean) -> scale
age_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='mean')),
    ('scale',  StandardScaler())
])

# 2. Pipeline for Fever: impute (median) -> scale
fever_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale',  StandardScaler())
])

# 3. Combine in ColumnTransformer
ct = ColumnTransformer(transformers=[
    ('age_pipe',   age_pipe,   ['Age']),
    ('fever_pipe', fever_pipe, ['Fever']),
    ('gender_ohe', OneHotEncoder(), ['Gender'])
])

# 4. Fit-transform and verify no NaNs remain
result = ct.fit_transform(df4)
print('Transformed result:')
print(result)

print('\nFeature names:', ct.get_feature_names_out())
print('\nAny NaN remaining in output?', np.isnan(result.astype(float)).any())
print('\nBoth Age and Fever missing values were imputed AND scaled in a')
print('single fit_transform() call, while Gender was one-hot encoded —')
print('all three transformations ran together, correctly combined.')

---
## Section 5: ColumnTransformer + Model in a Full Pipeline

In [ ]:
# ============================================================
# ANSWER 5: Full Preprocessing + Model Pipeline
# ============================================================

np.random.seed(10)
n = 150
df5 = pd.DataFrame({
    'Age':     np.random.randint(20, 65, n),
    'Salary':  np.random.randint(25000, 150000, n),
    'City':    np.random.choice(['Mumbai', 'Delhi', 'Pune'], n),
})
df5['Buys'] = ((df5['Salary'] > 70000) & (df5['Age'] < 50)).astype(int)

# 1 & 2: split
X = df5[['Age', 'Salary', 'City']]
y = df5['Buys']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# 3: ColumnTransformer
ct = ColumnTransformer(transformers=[
    ('scale', StandardScaler(), ['Age', 'Salary']),
    ('ohe',   OneHotEncoder(), ['City'])
])

# 4: full pipeline
full_pipeline = Pipeline([
    ('preprocessing', ct),
    ('model', LogisticRegression())
])

# 5: fit and score
full_pipeline.fit(X_train, y_train)
accuracy = full_pipeline.score(X_test, y_test)

print(f'Test accuracy: {accuracy:.4f}')
print('\nThe Pipeline ran ColumnTransformer.fit_transform on X_train,')
print('then LogisticRegression.fit, all from a single .fit() call.')
print('Calling .score() automatically applied the SAME fitted scaler/encoder')
print('to X_test before evaluating — zero risk of data leakage.')

---
## Section 6: Selecting Columns Automatically with make_column_selector

In [ ]:
# ============================================================
# ANSWER 6: make_column_selector
# ============================================================

df6 = pd.DataFrame({
    'Age':       [22, 35, 58, 41, 29],
    'Income':    [25000, 60000, 120000, 85000, 40000],
    'City':      ['Mumbai', 'Delhi', 'Pune', 'Mumbai', 'Delhi'],
    'Payment':   ['Card', 'UPI', 'Cash', 'UPI', 'Card']
})

# 1 & 2: build ColumnTransformer with automatic dtype-based selection
ct = ColumnTransformer(transformers=[
    ('num', StandardScaler(),
        make_column_selector(dtype_include='number')),
    ('cat', OneHotEncoder(),
        make_column_selector(dtype_include='object'))
])

# 3: fit-transform
result = ct.fit_transform(df6)
print('Transformed result:')
print(result)

print('\nGenerated feature names:')
print(ct.get_feature_names_out())

# 4: which columns each selector picked up
numeric_selector = make_column_selector(dtype_include='number')
categorical_selector = make_column_selector(dtype_include='object')
print('\nNumeric columns auto-selected:', numeric_selector(df6))
print('Categorical columns auto-selected:', categorical_selector(df6))

print('\nNo column names were hard-coded — make_column_selector inspected')
print('the dtypes of df6 and routed each column to the right transformer.')

---
## Section 7: Mini End-to-End Project — Mixed Dataset Preprocessing

In [ ]:
# ============================================================
# ANSWER 7: Full Mixed-Type Preprocessing Pipeline
# ============================================================

raw = pd.DataFrame({
    'Age':           [25, 40, 33, 55, 29, 47, 38, 60],
    'Fever':         [98.6, 101.2, np.nan, 99.5, 100.1, np.nan, 98.9, 102.0],
    'Gender':        ['Male', 'Female', 'Female', 'Male',
                        'Female', 'Male', 'Female', 'Male'],
    'City':          ['Mumbai', 'Delhi', 'Pune', 'Mumbai',
                        'Delhi', 'Pune', 'Mumbai', 'Delhi'],
    'Cough_Severity':['Mild', 'Strong', 'Mild', 'Strong',
                        'Mild', 'Strong', 'Mild', 'Strong'],
    'Has_Disease':   ['No', 'Yes', 'No', 'Yes', 'No', 'Yes', 'No', 'Yes']
})

# Step 1: split into X and y
X = raw.drop(columns=['Has_Disease'])
y = raw['Has_Disease']

# Step 2: train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42)

# Step 3: build the ColumnTransformer with 4 parts
fever_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='mean')),
    ('scale',  StandardScaler())
])

ct = ColumnTransformer(transformers=[
    ('age_scale',   StandardScaler(),                                ['Age']),
    ('fever_pipe',  fever_pipe,                                      ['Fever']),
    ('gender_city', OneHotEncoder(drop='first'),                     ['Gender', 'City']),
    ('cough_ord',   OrdinalEncoder(categories=[['Mild', 'Strong']]),  ['Cough_Severity'])
])

# Step 4: fit on training data only, transform both sets
ct.fit(X_train)
X_train_enc = ct.transform(X_train)
X_test_enc  = ct.transform(X_test)

# Step 5: encode y separately with LabelEncoder
y_encoder = LabelEncoder()
y_encoder.fit(y_train)
y_train_enc = y_encoder.transform(y_train)
y_test_enc  = y_encoder.transform(y_test)

# Step 6: print results
feature_names = ct.get_feature_names_out()
print('Feature names after ColumnTransformer:')
print(feature_names)

print('\nX_train (encoded):')
print(pd.DataFrame(X_train_enc, columns=feature_names).round(3))

print('\nX_test (encoded):')
print(pd.DataFrame(X_test_enc, columns=feature_names).round(3))

print('\ny_train (encoded):', y_train_enc)
print('y_test (encoded): ', y_test_enc)
print('Label classes:', y_encoder.classes_)

print('\nAll four column groups — Age, Fever (with missing values),')
print('Gender+City, and Cough_Severity — were transformed correctly in')
print('a single ColumnTransformer, fit only on training data.')

print('\nAmol Jagtap | amoljagtap3001@gmail.com')

---
## Summary & Quick Revision

| Concept | What You Learned |
|---|---|
| ColumnTransformer | Applies different transformers to different column subsets, then combines results |
| Syntax | `ColumnTransformer([('name', transformer, [columns]), ...])` |
| remainder='drop' | Default — unlisted columns are removed |
| remainder='passthrough' | Unlisted columns kept unchanged, appended at the end |
| Nested Pipeline | Use a Pipeline (impute -> scale) as a single transformer for one column |
| Full Pipeline | ColumnTransformer + model chained together — one fit() call |
| make_column_selector | Auto-selects columns by dtype instead of naming them manually |
| Fit rule | Always fit on training data only, transform on test data |

---
> **Notebook by:** Amol Jagtap | amoljagtap3001@gmail.com  
> **Topic:** Day 28 — Column Transformer